In [1]:
import numpy as np
import xarray as xr
import csv
import pandas as pd
import os
import netCDF4 as nc4
import sys
import xesmf

import cartopy.crs as ccrs

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure


def make_se_regridder(weight_file, regrid_method="conserved"):
    weights = xr.open_dataset(weight_file)
    in_shape = weights.src_grid_dims.load().data

    # Since xESMF expects 2D vars, we'll insert a dummy dimension of size-1
    if len(in_shape) == 1:
        in_shape = [1, in_shape.item()]

    # output variable shape
    out_shape = weights.dst_grid_dims.load().data.tolist()[::-1]

    #print(in_shape, out_shape)

    #Some prep to get the bounds:
    lat_b_out = np.zeros(out_shape[0]+1)
    lon_b_out = weights.xv_b.data[:out_shape[1]+1, 0]
    lat_b_out[:-1] = weights.yv_b.data[np.arange(out_shape[0])*out_shape[1],0]
    lat_b_out[-1] = weights.yv_b.data[-1,-1]

    dummy_in = xr.Dataset(
        {
            "lat": ("lat", np.empty((in_shape[0],))),
            "lon": ("lon", np.empty((in_shape[1],))),
            "lat_b": ("lat_b", np.empty((in_shape[0] + 1,))),
            "lon_b": ("lon_b", np.empty((in_shape[1] + 1,))),
        }
    )
    dummy_out = xr.Dataset(
        {
            "lat": ("lat", weights.yc_b.data.reshape(out_shape)[:, 0]),
            "lon": ("lon", weights.xc_b.data.reshape(out_shape)[0, :]),
            "lat_b": ("lat_b", lat_b_out),
            "lon_b": ("lon_b", lon_b_out),
        }
    )

    regridder = xesmf.Regridder(
        dummy_in,
        dummy_out,
        weights=weight_file,
        method=regrid_method,#"conservative_normed",
        #method="bilinear",
        reuse_weights=True,
        periodic=True,
    )
    return regridder



fates_file = '/datalake/NS9188K/share/jessien/feb26/ne16_transient_1900.2026-02-06/run/ne16_transient_1900.2026-02-06.2020.nc'
fates = xr.open_dataset(fates_file, decode_times=False)

weight_file = "/datalake/NS9560K/diagnostics/land_xesmf_diag_data/map_ne16pg3_to_1.9x2.5_nomask_scripgrids_c250425.nc"
regridder = make_se_regridder(weight_file=weight_file)

FATES_IGN = fates['FATES_IGNITIONS'].sum(dim='fates_levpft').mean(dim='time')
vegc_PF_se = xr.DataArray(FATES_IGN)
vegc_pf = rf.regrid_se_data(regridder, vegc_PF_se)

km2_to_m2 = 1e6
kg_to_Gt = 1e-12
area = fates['area']
area = xr.DataArray(area)
area = rf.regrid_se_data(regridder, area)

landfrac = fates['landfrac']
landfrac = xr.DataArray(landfrac)
landfrac = rf.regrid_se_data(regridder, landfrac)

fatesfrac = fates['FATES_FRACTION'].mean(dim='time')
fatesfrac = xr.DataArray(fatesfrac)
fatesfrac = rf.regrid_se_data(regridder, fatesfrac)

area_land = area * landfrac * km2_to_m2 

total_globe = (vegc_pf * area_land * kg_to_Gt * fatesfrac).sum(dim=['lat','lon']).values
print(total_globe)

ModuleNotFoundError: No module named 'ESMF'